In [1]:
import matplotlib.pyplot as plt
from glob import glob
import matplotlib.patches as mpatches
import cosima_cookbook as cc
import numpy as np
from matplotlib.gridspec import GridSpec
import matplotlib.ticker as mticker
import cmocean.cm as cm
import xarray as xr
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cft
import gc
from scipy.interpolate import griddata
from matplotlib import ticker
import scipy.io as sio
import seawater as sw
import seaborn as sns
import matplotlib.patheffects as pe
from scipy import stats
import matplotlib.patheffects as mpe
from tqdm import tqdm

In [2]:
# Set figures properties
# sns.set_style('whitegrid')
# sns.reset_orig()
plt.rcParams['font.size'] = 14
plt.rcParams['font.weight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titleweight'] = 'bold'

## Import SST

In [4]:
path = '/g/data/v45/fs3401/AmelieMeyer_Agulhas_meander/data/'
time_sel = slice('1993-01-01', '2019-05-31')
SST  = xr.open_dataset(path+'sst_19820101-20211231_fromChrisC.nc').sst.load()[:,0]
SST = SST.sel().sel(time=time_sel)
lon_SST, lat_SST = SST.lon, SST.lat

In [7]:
SST_mean = SST.mean('time').compute()

## Import SLA - There is an issue going on at the moment with ua8 data

In [8]:
# #########################################      CMEMS
# path = '/g/data/ua8/CMEMS_SeaLevel/DT-2021/'
# # Domain
# yt_sel = slice(-51, -30.5)
# xt_sel = slice(8, 76)

# # <year>/<month>/<files>

# # Filenames are:

# # dt_global_allsat_phy_l4_YYYYMMDD_<production_date>.nc

# # Period
# year = np.arange(1993,2019,1)
# month = np.arange(1,13,1)
# data = np.array([]) 
# for i in range(len(year)):
#     for m in range(len(month)):
#         datum_year = glob(path + str(year[i]) + '/' + str(month[m]).zfill(2) + '/dt_global_allsat_phy_l4_' + 
#                                   str(year[i])+'*.nc')
#         datum_year.sort()
#         data = np.append(data, datum_year)

In [11]:
# from functools import partial
# def _preprocess(ds, lon_bnds, lat_bnds):
#     return ds.sel(longitude = lon_bnds, latitude = lat_bnds)

# partial_func = partial(_preprocess, lon_bnds=xt_sel, lat_bnds=yt_sel)
# data_CMEMS = xr.open_mfdataset(list(data), preprocess=partial_func,
#                                drop_variables = ['crs', 'lat_bnds', 'lon_bnds', 'err_sla', 'ugosa', 'err_ugosa', 
#                                                  'vgosa', 'err_vgosa', 'adt', 'ugos', 'vgos', 'flag_ice', 'tpa_correction'])
# SLA = data_CMEMS.sla.load()
# Ln, Lt = SLA.longitude, SLA.latitude

## Importing ERA5

In [17]:
path_Ta = '/g/data/rt52/era5/single-levels/reanalysis/2t/'
path_u = '/g/data/rt52/era5/single-levels/reanalysis/10u/'
path_v = '/g/data/rt52/era5/single-levels/reanalysis/10v/'
path_mtpr = '/g/data/rt52/era5/single-levels/reanalysis/mtpr/'
path_mer  = '/g/data/rt52/era5/single-levels/reanalysis/mer/'
path_sshf = '/g/data/rt52/era5/single-levels/reanalysis/sshf/'
path_slhf = '/g/data/rt52/era5/single-levels/reanalysis/slhf/'
path_tcc  = '/g/data/rt52/era5/single-levels/reanalysis/tcc/'
year   = np.arange(1993,2020,1)

files_Ta, files_tcc = np.array([]), np.array([])
files_u, files_v = np.array([]), np.array([])
files_mtpr, files_mer = np.array([]), np.array([])
files_sshf, files_slhf = np.array([]), np.array([])
## The files are in different directories
for i in range(len(year)):
    file_Ta = glob(path_Ta + str(year[i]) + '/*')
    file_u, file_v = glob(path_u + str(year[i]) + '/*'), glob(path_v + str(year[i]) + '/*')
    file_mtpr = glob(path_mtpr + str(year[i]) + '/*')
    file_mer = glob(path_mer + str(year[i]) + '/*')
    file_sshf, file_slhf = glob(path_sshf + str(year[i]) + '/*'), glob(path_slhf + str(year[i]) + '/*')
    file_tcc = glob(path_tcc + str(year[i]) + '/*')
    
    file_Ta.sort()
    file_u.sort()
    file_v.sort()
    file_mtpr.sort()
    file_mer.sort()
    file_sshf.sort()
    file_slhf.sort()
    file_tcc.sort()
    
    files_Ta = np.append(files_Ta, file_Ta)
    files_u  = np.append(files_u, file_u)
    files_v  = np.append(files_v, file_v)
    files_mtpr = np.append(files_mtpr, file_mtpr)
    files_mer = np.append(files_mer, file_mer)
    files_sshf = np.append(files_sshf, file_sshf)
    files_slhf = np.append(files_slhf, file_slhf)
    files_tcc  = np.append(files_tcc, file_tcc)

In [18]:
from functools import partial
def _preprocess(ds, lon_bnds, lat_bnds):
    return ds.sel(longitude = lon_bnds, latitude = lat_bnds)

yt_sel = slice(-30.5, -51)
xt_sel = slice(8, 76)
partial_func = partial(_preprocess, lon_bnds=xt_sel, lat_bnds=yt_sel) 

In [19]:
data_Ta = xr.open_mfdataset(list(files_Ta), preprocess=partial_func)
data_u  = xr.open_mfdataset(list(files_u), preprocess=partial_func)  
data_v  = xr.open_mfdataset(list(files_v), preprocess=partial_func)  
data_mtpr = xr.open_mfdataset(list(files_mtpr), preprocess=partial_func)
data_mer  = xr.open_mfdataset(list(files_mer), preprocess=partial_func)
data_sshf = xr.open_mfdataset(list(files_sshf), preprocess=partial_func)  
data_slhf = xr.open_mfdataset(list(files_slhf), preprocess=partial_func)  
data_tcc  = xr.open_mfdataset(list(files_tcc), preprocess=partial_func)  

In [20]:
time_sel   = slice('1993-01-01', '2019-05-31')

Ta         = data_Ta.t2m.sel(time=time_sel) - 273.15
temp_daily = Ta.resample(time='1D').mean('time').compute()
##############################################################
u_atm       = data_u.u10.sel(time=time_sel)
u_atm_daily = u_atm.resample(time='1D').mean('time').compute()
##############################################################
v_atm       = data_v.v10.sel(time=time_sel)
v_atm_daily = v_atm.resample(time='1D').mean('time').compute()
##############################################################
mtpr       = data_mtpr.mtpr.sel(time=time_sel) ## mean total precipitation rate
mtpr_daily = mtpr.resample(time='1D').mean('time').compute()*86400 # This multiplication converts the s-1 to day-1
##############################################################
mer       = data_mer.mer.sel(time=time_sel) ## Mean evaporation rate
mer_daily = mer.resample(time='1D').mean('time').compute()*86400 # This multiplication converts the s-1 to day-1
##############################################################
sshf       = data_sshf.sshf.sel(time=time_sel) ## surface sensible heat flux
sshf_daily = sshf.resample(time='1D').mean('time').compute()/3600 
##############################################################
slhf       = data_slhf.slhf.sel(time=time_sel) ## surface latent heat flux
slhf_daily = slhf.resample(time='1D').mean('time').compute()/3600
##############################################################
cloud       = data_tcc.tcc.sel(time=time_sel)
cloud_daily = cloud.resample(time='1D').mean('time').compute()*100 ## I mulitply to a hundred to obtain in percentage

In [21]:
lon_ERA5, lat_ERA5 = u_atm.longitude, u_atm.latitude

wind_speed = np.array([np.sqrt(u_atm_daily[t]**2 + v_atm_daily[t]**2) for t in tqdm(range(len(u_atm_daily)))])

100%|██████████| 9647/9647 [00:20<00:00, 475.25it/s]


In [23]:
Ta_mean       = temp_daily.mean('time').compute()
wind_speed_mean = np.nanmean(wind_speed, axis = 0)
mtpr_mean       = mtpr_daily.mean('time').compute()
mer_mean        = mer_daily.mean('time').compute()
sshf_mean       = sshf_daily.mean('time').compute()
slhf_mean       = slhf_daily.mean('time').compute()
tcc_mean        = cloud_daily.mean('time').compute()

## Saving the fields above

In [24]:
ds = xr.Dataset(data_vars=dict(Ta_mean            = (["lat_ERA5", "lon_ERA5"], Ta_mean.data),
                               wind_speed_mean    = (["lat_ERA5", "lon_ERA5"], wind_speed_mean),
                               precipitation_mean = (["lat_ERA5", "lon_ERA5"], mtpr_mean.data),
                               evaporation_mean   = (["lat_ERA5", "lon_ERA5"], mer_mean.data),
                               sshf_mean          = (["lat_ERA5", "lon_ERA5"], sshf_mean.data),
                               slhf_mean          = (["lat_ERA5", "lon_ERA5"], slhf_mean.data),
                               tcc_mean           = (["lat_ERA5", "lon_ERA5"], tcc_mean.data),
                               SST_mean           = (["lat_SST", "lon_SST"], SST_mean.data)
                              ),
                        
                coords=dict(lat_ERA5=(["lat_ERA5"], lat_ERA5.data),
                            lon_ERA5=(["lon_ERA5"], lon_ERA5.data),
                            lat_SST=(["lat_SST"], lat_SST.data),
                            lon_SST=(["lon_SST"], lon_SST.data)),
                attrs=dict(description="Time-mean between Jan 1993 and May 2019."))

path_save = '/g/data/jk72/fs3401/Desktop/Agulhas_meander/output/'
ds.to_netcdf(path_save+'mean_26years.nc')